# Pet the Dog, Feed the Soul: Reproducing All Analyses (v1.36)

This notebook is the **single entry point** for reproducing the analyses reported in *Pet the Dog, Feed the Soul: Investigating the Psychological Mechanisms of Animal-Assisted Interventions for State Anxiety*.

## Instructions
1. Install dependencies from `requirements.txt` (see `README.md`).
2. Launch Jupyter from the repository root directory.
3. Choose **Run All** to execute the offline tabular checks (Analysis 1 self-report models, Analysis 2 multimodal dose models and residual correlations, and zero-shot MLLM prediction replay) and verify manuscript figure assets.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
assert (ROOT / "scripts" / "reproduce.py").is_file(), "Please start Jupyter at the repository root."
sys.path.insert(0, str(ROOT / "scripts"))

from reproduce import run, verify_assets

## 1. Analysis 1 (Self-Report) & Analysis 2 (Multimodal Features + MLLM Replay)

Running `run()` executes the offline verification pipeline on the de-identified tabular datasets in `data/`:
- **Analysis 1 (n = 100):** STAI total, STAI-AA, and STAI-AP descriptives, reliable relief rates ($\Delta\text{STAI} \ge 7.33$), and baseline-adjusted exposure-dose ANCOVAs with departure contrasts.
- **Analysis 2 (n = 92 / 79 / 100):** Exposure-dose ANCOVAs and BH-FDR control across all 19 facial, acoustic, thermal, and cardiovascular features, plus the 114 baseline-adjusted residual correlations.
- **Zero-Shot MLLM Replay (184 clips):** MAE, $R^2$, and pooled Pearson $r$ for Experiments 1–4 from frozen predictions in `data/model_predictions/`.

In [ ]:
effects, mllm_scores = run()
print("Analysis 1 & 2 exposure-dose effects:")
display(effects.round(4))
print("Zero-shot MLLM replay metrics (Experiments 1–4):")
display(mllm_scores.round(4))

## 2. Verify Manuscript Figure Assets & Inspect Generated Redraws

Verifies SHA-256 checksums for the archived manuscript figure assets and displays any newly redrawn figures saved to `generated/figures/`.

In [ ]:
asset_table = pd.DataFrame(verify_assets())[["figure", "asset", "path"]]
display(asset_table)

for fig_path in sorted((ROOT / "generated" / "figures").glob("*.png")):
    display(Image(filename=str(fig_path)))

## 3. Modular Downstream & Upstream Analysis Notebooks (Optional / Reference)

For full deep-dive scripts and upstream pipelines corresponding to each section of the manuscript:
- **Analysis 1 (Mediation, Parallel Pathways, Moderation):** `statistical_analysis/study1_self_report_analysis.py`
- **Analysis 2 (Multimodal Statistics):** `statistical_analysis/study2_multimodal_statistics.py`
- **Analysis 2 (Nested CV Component Identification & Anxiety Regression — Figure 8, Supplementary Figures S1–S2):** `statistical_analysis/03_study2_multimodal_machine_learning.ipynb`
- **Analysis 2 (MLLM Prediction & Explanation Cue Coding — Supplementary Figure S3):** `statistical_analysis/04_study2_mllm_analysis.ipynb`
- **Publication Figures (Figures 1–8, Supplementary Figures S1–S3):** `chart/manuscript_figures.ipynb`
- **Upstream Feature Extraction & Raw MLLM Inference (requires restricted raw recordings / API keys):** `feature_extraction/` and `mllm.ipynb`